# Lesson 2.4: Conditional Probability

Read conditional probabilities from the lesson's table of 200 students (studied or not, passed or not) and show that P(pass given studied) is not P(studied given pass).

Part of *Statistics from Scratch*. Run each cell with **Shift + Enter**, top to bottom.

If a package is missing on your computer, run the next cell once. In Google Colab everything is already installed.

In [ ]:
%pip install -q numpy scipy pandas statsmodels matplotlib

## 1. Build the two-way table

We list the 200 students as (group, result) pairs, then pandas crosstab counts them into the same table you saw in the lesson. margins=True adds the totals.

In [1]:
import pandas as pd

rows = ([('Studied', 'Passed')] * 90 + [('Studied', 'Failed')] * 10
        + [('No study', 'Passed')] * 40 + [('No study', 'Failed')] * 60)
df = pd.DataFrame(rows, columns=['group', 'result'])

table = pd.crosstab(df['group'], df['result'], margins=True)
print(table)

result    Failed  Passed  All
group                        
No study      60      40  100
Studied       10      90  100
All           70     130  200


## 2. P(pass given studied), two ways

First restrict to the Studied row and take the share who passed. Then use the formula P(A and B) / P(B). Both routes must agree.

In [2]:
n = len(df)
studied = table.loc['Studied', 'All']
both = table.loc['Studied', 'Passed']

print(f'row method : {both} / {studied} = {both / studied:.3f}')
print(f'formula     : {both / n:.3f} / {studied / n:.3f} = {(both / n) / (studied / n):.3f}')

row method : 90 / 100 = 0.900
formula     : 0.450 / 0.500 = 0.900


## 3. The reverse question is different

P(studied given pass) restricts to the Passed column instead. The numbers differ, so never swap the two. Normalising the table by row or by column shows both at once.

In [3]:
passed = table.loc['All', 'Passed']
print(f'P(pass | studied) = {both / studied:.3f}')
print(f'P(studied | pass) = {both} / {passed} = {both / passed:.3f}')

print('\nrow percentages (condition on the row):')
print(pd.crosstab(df['group'], df['result'], normalize='index').round(3))

P(pass | studied) = 0.900
P(studied | pass) = 90 / 130 = 0.692

row percentages (condition on the row):
result    Failed  Passed
group                   
No study     0.6     0.4
Studied      0.1     0.9


## Try this

- Find P(fail | no study) from the table, and predict how it relates to P(pass | no study) before you check.
- Change 90 passers to 70 and 40 non-study passers to 60. Does P(pass | studied) still beat P(pass)?

## The same idea in R, SPSS and Stata

**R**

```r
tab <- matrix(c(90, 10, 40, 60), nrow = 2, byrow = TRUE,
              dimnames = list(c("Studied", "No study"), c("Passed", "Failed")))
addmargins(tab)
tab[1, 1] / sum(tab[1, ])         # P(pass | studied)
tab[1, 1] / sum(tab[, 1])         # P(studied | pass)
prop.table(tab, 1)                # row proportions
```

**SPSS** (Data > Weight Cases (weight by freq); Analyze > Descriptive Statistics > Crosstabs, then Cells: tick Row and Column percentages)

```
DATA LIST FREE / group result freq.
BEGIN DATA
1 1 90
1 2 10
2 1 40
2 2 60
END DATA.
VALUE LABELS group 1 'Studied' 2 'No study'
  /result 1 'Passed' 2 'Failed'.
WEIGHT BY freq.
CROSSTABS /TABLES=group BY result
  /CELLS=COUNT ROW COLUMN.
```

**Stata**

```
clear
input group result freq
1 1 90
1 2 10
2 1 40
2 2 60
end
label define g 1 "Studied" 2 "No study"
label values group g
label define r 1 "Passed" 2 "Failed"
label values result r
tabulate group result [fweight = freq], row column
```